# DriverFlow — Annotation Tool Launcher

This notebook sets up and launches the DriverFlow annotation tool in Google Colab.

**Requirements:**
1. GitHub Personal Access Token (PAT) with `repo` scope — get one at https://github.com/settings/tokens
2. (Optional) ngrok auth token for public URL — get one at https://dashboard.ngrok.com/get-started/your-authtoken

**Instructions:**
1. Fill in your credentials in the next cell
2. Click `Runtime → Run all` to launch the full tool

In [ ]:
# === CONFIGURE THESE VALUES ===

GITHUB_TOKEN = ""          # Your GitHub PAT (required)
GITHUB_USERNAME = ""       # Your GitHub username (required)
NGROK_TOKEN = ""           # Your ngrok auth token (optional, for public URL)

# =================================

if not GITHUB_TOKEN or not GITHUB_USERNAME:
    print("⚠️  SETUP REQUIRED: Fill in GITHUB_TOKEN and GITHUB_USERNAME above, then re-run this cell.")
else:
    print("✓ Credentials configured. Proceeding with setup...")

## Step 1: Install GroundingDINO with CUDA patch

In [ ]:
%%bash
set -e
cd /content

if [ ! -d "GroundingDINO" ]; then
  echo "Cloning GroundingDINO..."
  git clone https://github.com/IDEA-Research/GroundingDINO.git
else
  echo "GroundingDINO already present."
fi

echo "Applying CUDA patch..."
CUDA_FILE="GroundingDINO/groundingdino/models/GroundingDINO/csrc/MsDeformAttn/ms_deform_attn_cuda.cu"
sed -i 's/value.type()/value.scalar_type()/g' "$CUDA_FILE" 2>/dev/null || true
sed -i 's/value.scalar_type().is_cuda()/value.is_cuda()/g' "$CUDA_FILE" 2>/dev/null || true

echo "Installing GroundingDINO..."
cd GroundingDINO
pip install -q -e .
pip install -q supervision transformers==4.38.2

echo "✓ GroundingDINO installed."

## Step 2: Download model weights

In [ ]:
%%bash
mkdir -p /content/weights

if [ ! -f "/content/weights/groundingdino_swint_ogc.pth" ]; then
  echo "Downloading model weights (may take a few minutes)..."
  wget -q -O /content/weights/groundingdino_swint_ogc.pth \
    https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
  echo "✓ Weights downloaded."
else
  echo "✓ Weights already present."
fi

## Step 3: Clone DriverFlow repo and install backend dependencies

In [ ]:
import os
import subprocess

token = GITHUB_TOKEN
username = GITHUB_USERNAME

repo_url = f"https://{token}@github.com/{username}/DriverFlow.git"

if not os.path.exists("/content/DriverFlow"):
    print("Cloning DriverFlow repository...")
    subprocess.run(["git", "clone", repo_url, "/content/DriverFlow"], check=True)
    print("✓ Repository cloned.")
else:
    print("Pulling latest changes...")
    subprocess.run(["git", "-C", "/content/DriverFlow", "pull"], check=True)
    print("✓ Repository updated.")

print("Installing backend dependencies...")
subprocess.run(
    ["pip", "install", "-q", "-r", "/content/DriverFlow/backend/requirements.txt"],
    check=True
)
print("✓ Dependencies installed.")

## Step 4: Start the backend server

In [ ]:
import subprocess
import time
import os

print("Starting FastAPI backend...")
proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/DriverFlow/backend",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)
print("✓ Backend started on port 8000.")

## Step 5: Expose via ngrok or Colab proxy

In [ ]:
import subprocess

ngrok_token = NGROK_TOKEN

if ngrok_token:
    # Use ngrok for public URL
    from pyngrok import ngrok, conf

    try:
        ngrok.kill()
    except:
        pass

    conf.get_default().auth_token = ngrok_token
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url

    print(f"""
╔══════════════════════════════════════════╗
║      🚀 DriverFlow is LIVE! 🚀           ║
║                                          ║
║  Open this URL in your browser:          ║
║  {public_url:<40}║
║                                          ║
╚══════════════════════════════════════════╝
    """)

else:
    # Use Colab's built-in proxy
    from google.colab.output import eval_js

    url = eval_js("google.colab.kernel.proxyPort(8000)")

    print(f"""
╔══════════════════════════════════════════╗
║      🚀 DriverFlow is LIVE! 🚀           ║
║                                          ║
║  Open this URL in your browser:          ║
║  {url:<40}║
║                                          ║
║  Note: Colab proxy may auto-logout.      ║
║  For persistent access, use ngrok token. ║
║                                          ║
╚══════════════════════════════════════════╝
    """)

print("The notebook will remain running. Do not close this cell.")